# 02 - GANs: Intuition and Basics (OPTIONAL)

## Learning Objectives
- Understand the Generator vs Discriminator adversarial game
- Implement a simple GAN for 2D data
- Know the key training challenges (mode collapse, instability)

## Prerequisites
- DL100-DL200 (neural networks, training loops)
- Notebook 01 (autoencoders)

## Table of Contents
1. [GAN Concept](#1)
2. [The Minimax Game](#2)
3. [Simple GAN: 2D Gaussian](#3)
4. [GAN on MNIST](#4)
5. [Training Challenges](#5)
6. [Exercises](#6)
7. [Common Mistakes & Debugging Tips](#7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import sys; sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed
set_global_seed(42)

<a id='1'></a>
## 1. GAN Concept

A **Generative Adversarial Network** consists of two networks competing:

- **Generator (G):** Takes random noise $z \sim \mathcal{N}(0,1)$ and produces fake samples
- **Discriminator (D):** Classifies samples as real or fake

Analogy: G is a counterfeiter, D is a detective. Both improve through competition.

```
Random Noise z → [Generator] → Fake Data
                                    ↓
Real Data ──────────────────→ [Discriminator] → Real/Fake?
```

<a id='2'></a>
## 2. The Minimax Game

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- **D wants to maximize:** correctly classify real as real, fake as fake
- **G wants to minimize:** fool D into classifying fake as real
- At equilibrium: $D(x) = 0.5$ (can't tell real from fake)

<a id='3'></a>
## 3. Simple GAN: 2D Gaussian

In [ ]:
# Target: 2D Gaussian distribution
real_mean = torch.tensor([3.0, 3.0])
real_std = 0.5
n_samples = 1000
real_data = real_mean + real_std * torch.randn(n_samples, 2)

# Generator: noise → 2D points
class Generator(nn.Module):
    def __init__(self, noise_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, z):
        return self.net(z)

# Discriminator: 2D points → real/fake probability
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

noise_dim = 16
G = Generator(noise_dim)
D = Discriminator()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
# Training loop
loader = DataLoader(TensorDataset(real_data), batch_size=128, shuffle=True)
g_losses, d_losses = [], []

for epoch in range(200):
    for (real_batch,) in loader:
        bs = real_batch.size(0)
        real_labels = torch.ones(bs, 1)
        fake_labels = torch.zeros(bs, 1)
        
        # --- Train Discriminator ---
        z = torch.randn(bs, noise_dim)
        fake_data = G(z).detach()
        
        d_real = loss_fn(D(real_batch), real_labels)
        d_fake = loss_fn(D(fake_data), fake_labels)
        d_loss = (d_real + d_fake) / 2
        
        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()
        
        # --- Train Generator ---
        z = torch.randn(bs, noise_dim)
        fake_data = G(z)
        g_loss = loss_fn(D(fake_data), real_labels)  # fool D
        
        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()
    
    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1} | D Loss: {d_loss.item():.3f} | G Loss: {g_loss.item():.3f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Generated vs real
with torch.no_grad():
    z = torch.randn(500, noise_dim)
    generated = G(z).numpy()
axes[0].scatter(real_data[:500, 0], real_data[:500, 1], alpha=0.4, label='Real', s=10)
axes[0].scatter(generated[:, 0], generated[:, 1], alpha=0.4, label='Generated', s=10)
axes[0].legend(); axes[0].set_title('Real vs Generated'); axes[0].grid(True, alpha=0.3)

# Loss curves
axes[1].plot(g_losses, label='Generator', alpha=0.7)
axes[1].plot(d_losses, label='Discriminator', alpha=0.7)
axes[1].legend(); axes[1].set_title('GAN Losses'); axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch')

plt.tight_layout(); plt.show()

<a id='4'></a>
## 4. GAN on MNIST

A simple MNIST GAN. Note: this may train slowly on CPU.

In [ ]:
# Load MNIST (or fallback)
try:
    from torchvision import datasets, transforms
    mnist = datasets.MNIST(root='../../data/raw', train=True, download=True,
                           transform=transforms.Compose([transforms.ToTensor(),
                                                         transforms.Normalize([0.5], [0.5])]))
    mnist_loader = DataLoader(mnist, batch_size=128, shuffle=True)
    MNIST_OK = True
    img_dim = 784  # 28*28
except Exception:
    MNIST_OK = False
    print("MNIST not available. Using synthetic data demo above.")

In [ ]:
if MNIST_OK:
    # Simple MLP GAN for MNIST
    latent_dim = 64
    G_mnist = nn.Sequential(
        nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
        nn.Linear(256, 512), nn.LeakyReLU(0.2),
        nn.Linear(512, img_dim), nn.Tanh()
    )
    D_mnist = nn.Sequential(
        nn.Linear(img_dim, 512), nn.LeakyReLU(0.2),
        nn.Linear(512, 256), nn.LeakyReLU(0.2),
        nn.Linear(256, 1)
    )
    opt_g = torch.optim.Adam(G_mnist.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D_mnist.parameters(), lr=2e-4, betas=(0.5, 0.999))
    
    # Train for just 5 epochs (demo)
    for epoch in range(5):
        for real_imgs, _ in mnist_loader:
            bs = real_imgs.size(0)
            real_flat = real_imgs.view(bs, -1)
            
            # Train D
            z = torch.randn(bs, latent_dim)
            d_loss = (loss_fn(D_mnist(real_flat), torch.ones(bs,1)) +
                      loss_fn(D_mnist(G_mnist(z).detach()), torch.zeros(bs,1))) / 2
            opt_d.zero_grad(); d_loss.backward(); opt_d.step()
            
            # Train G
            z = torch.randn(bs, latent_dim)
            g_loss = loss_fn(D_mnist(G_mnist(z)), torch.ones(bs,1))
            opt_g.zero_grad(); g_loss.backward(); opt_g.step()
        
        print(f"Epoch {epoch+1}/5 | D: {d_loss.item():.3f} | G: {g_loss.item():.3f}")
    
    # Show generated samples
    with torch.no_grad():
        samples = G_mnist(torch.randn(16, latent_dim)).view(-1, 1, 28, 28)
    fig, axes = plt.subplots(2, 8, figsize=(12, 3))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i, 0].numpy() * 0.5 + 0.5, cmap='gray')
        ax.axis('off')
    plt.suptitle('Generated MNIST Samples (5 epochs)')
    plt.tight_layout(); plt.show()

<a id='5'></a>
## 5. Training Challenges

- **Mode collapse:** G produces limited variety (maps all z to few outputs)
- **Training instability:** D becomes too strong → G gradients vanish
- **Tips:** label smoothing (0.9 instead of 1.0), alternating update schedules
- **Wasserstein GAN (WGAN):** uses Earth Mover distance instead of BCE (more stable)

<a id='6'></a>
## 6. Exercises

**Exercise 1:** Modify the 2D GAN generator architecture (add layers) and observe output quality.

**Exercise 2:** Implement label smoothing (use 0.9 for real labels) and compare training stability.

**Exercise 3:** What happens if you train D much more than G? Try 5 D steps per 1 G step.

<a id='7'></a>
## 7. Common Mistakes & Debugging Tips

| Mistake | Fix |
|---------|-----|
| G loss goes to 0 quickly | D is too strong — reduce D capacity or learning rate |
| Mode collapse (all outputs same) | Add diversity to training, try WGAN |
| Forgetting `.detach()` on fake data when training D | G gets gradients it shouldn't |
| Using wrong labels | G wants D to output 1 (real) for fakes |